# SUBCPMK 3 — PN Tanjung Redeb 

Program mandiri untuk ekstraksi tabel ringkasan serta dukungan crawler situs resmi. File ZIP pengguna adalah **ringkasan PDF satu halaman**, belum dapat dinyatakan sebagai putusan lengkap resmi atau memenuhi syarat non-BHT. Hasil audit mencatat batasan itu.


## 1. Persiapan lingkungan
Jalankan seluruh sel berurutan. Google Colab: `Runtime → Run all`. ZIP standar dideteksi otomatis dari root proyek atau `/content`; untuk lokasi lain isi `ZIP_MANUAL`. Alur aman: berkas PDF yang sudah dipilih dimasukkan ke folder `pdf`.

In [15]:
%pip -q install requests beautifulsoup4 pymupdf pandas playwright

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
from urllib import robotparser
from collections import OrderedDict
import csv, io, re, time, shutil, zipfile, hashlib, json, os, unicodedata
import requests
from bs4 import BeautifulSoup
import pandas as pd
import fitz

def cari_root_proyek(start=None):
    start = (start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        if (folder / 'README_MULAI_DI_SINI.md').is_file():
            return folder
    return None

COLAB = Path('/content').is_dir()
PROJECT_ROOT = None if COLAB else cari_root_proyek()
ROOT = Path('/content/PN_Tanjung_Redeb') if COLAB else (PROJECT_ROOT / '03_SUBCPMK3_Crawler_dan_Ekstraksi' / 'PN_Tanjung_Redeb' if PROJECT_ROOT else Path.cwd() / 'PN_Tanjung_Redeb')
PDF_DIR = ROOT / 'pdf'
TXT_DIR = ROOT / 'txt'
OUT_DIR = ROOT / 'hasil_subcpmk3'
for folder in (PDF_DIR, TXT_DIR, OUT_DIR): folder.mkdir(parents=True, exist_ok=True)
ZIP_MANUAL = ''  # opsional: isi path ZIP secara manual jika deteksi otomatis gagal
zip_candidates = [Path(ZIP_MANUAL)] if ZIP_MANUAL else []
if COLAB:
    zip_candidates += [Path('/content/INPUT_DATASET_50_PDF_UNTUK_COLAB.zip'), Path('/content/50_Putusan_PN_Tanjung_Redeb_Narkotika_Tanpa_BHT.zip'), Path('/content/Narkotika.zip')]
elif PROJECT_ROOT:
    zip_candidates += [PROJECT_ROOT / 'INPUT_DATASET_50_PDF_UNTUK_COLAB.zip', PROJECT_ROOT / '02_SUBCPMK2_Dataset_GitHub' / 'Dataset-Narkotika_XXX_YYY' / 'Dataset' / 'Narkotika.zip']
zip_candidates += [Path.cwd() / 'INPUT_DATASET_50_PDF_UNTUK_COLAB.zip', Path.cwd() / 'Narkotika.zip']
ZIP_INPUT = next((p.resolve() for p in zip_candidates if p.is_file() and zipfile.is_zipfile(p)), None)
BASE_URL = 'https://putusan3.mahkamahagung.go.id'
START_URL = f'{BASE_URL}/direktori.html'
TARGET_URLS = [f'{BASE_URL}/direktori/index/pengadilan/pn-tanjung-redep/kategori/narkotika-dan-psikotropika-1/tahunjenis/putus/tahun/{tahun}.html' for tahun in (2025, 2024, 2023)]
PRINT_LIMIT = 15
print('Root proyek:', PROJECT_ROOT or '(Colab/tidak terdeteksi)')
print('Folder kerja bersama:', ROOT)
print('ZIP terpilih:', ZIP_INPUT or 'TIDAK DITEMUKAN')

Root proyek: C:\Users\marce\KULIAH\Smt7\InformationRetrival\UTS
Folder kerja bersama: C:\Users\marce\KULIAH\Smt7\InformationRetrival\UTS\03_SUBCPMK3_Crawler_dan_Ekstraksi\PN_Tanjung_Redeb
ZIP terpilih: C:\Users\marce\KULIAH\Smt7\InformationRetrival\UTS\INPUT_DATASET_50_PDF_UNTUK_COLAB.zip


## 2. Input PDF dari ZIP

Di Colab, unggah `INPUT_DATASET_50_PDF_UNTUK_COLAB.zip` ke `/content/` lewat panel Files. Di komputer lokal, notebook otomatis mencari ZIP di root UTS dan folder Dataset SUBCPMK 2. Gunakan `ZIP_MANUAL` hanya untuk lokasi lain.


In [17]:
if ZIP_INPUT is not None:
    src = Path(ZIP_INPUT)
    if not src.is_file():
        print('ZIP tidak ditemukan:', src)
    elif not zipfile.is_zipfile(src):
        print('Bukan ZIP valid:', src)
    else:
        imported = 0
        with zipfile.ZipFile(src) as z:
            for info in z.infolist():
                if info.is_dir() or not info.filename.lower().endswith('.pdf'):
                    continue
                safe_name = Path(info.filename).name
                if not safe_name: continue
                dest = PDF_DIR / safe_name
                if dest.exists() and dest.read_bytes() == z.read(info):
                    imported += 1
                    continue
                with z.open(info) as fin, open(dest,'wb') as fout:
                    shutil.copyfileobj(fin,fout)
                if dest.read_bytes()[:5] != b'%PDF-':
                    dest.unlink(missing_ok=True)
                    print('Bukan PDF valid, dilewati:',safe_name)
                else: imported += 1
        print('PDF tersedia dari ZIP:', imported)
else:
    print('ZIP dataset tidak ditemukan. Isi ZIP_MANUAL atau unggah INPUT_DATASET_50_PDF_UNTUK_COLAB.zip.')
print('PDF saat ini:', len(list(PDF_DIR.glob('*.pdf'))))

PDF tersedia dari ZIP: 50
PDF saat ini: 50


## 3. Crawler publik yang mematuhi pembatasan situs
Crawler aktif dengan titik masuk `https://putusan3.mahkamahagung.go.id/direktori.html`, lalu memprioritaskan direktori PN Tanjung Redeb kategori narkotika/psikotropika tahun 2023–2025. Script memeriksa `robots.txt`, membatasi laju permintaan, memvalidasi isi halaman detail, serta berhenti bila situs menolak akses (403, 429, 503, atau halaman verifikasi). **Tidak ada teknik melewati captcha/Cloudflare.**

Catatan: akses Colab ke Direktori MA bisa gagal. Dalam situasi itu, jalur pemrosesan PDF manual pada sel berikutnya tetap dapat berjalan.

In [18]:
AKTIFKAN_CRAWL = False  # fallback requests; crawler utama memakai Playwright pada sel berikutnya
BATAS_DOKUMEN_CRAWLER = 50
BATAS_HALAMAN_NAVIGASI = 40
TUNDA_DETIK = 3
SESSION = requests.Session()
SESSION.headers.update({'User-Agent':'Academic-IR-Research/1.0 (non-commercial; polite crawler)'})

ROBOTS = robotparser.RobotFileParser(BASE_URL + '/robots.txt')
ROBOTS_DICEK = False
ROBOTS_ERROR = ''
def boleh_akses(url):
    global ROBOTS_DICEK, ROBOTS_ERROR
    p = urlparse(url)
    if p.scheme != 'https' or p.netloc != 'putusan3.mahkamahagung.go.id':
        return False
    try:
        if not ROBOTS_DICEK:
            ROBOTS_DICEK = True
            ROBOTS.read()
        if ROBOTS_ERROR: return False
        return ROBOTS.can_fetch(SESSION.headers.get('User-Agent','Academic-IR-Research/1.0'), url)
    except Exception as exc:
        ROBOTS_ERROR = str(exc)[:160]
        print('Tidak dapat memeriksa robots.txt:',ROBOTS_ERROR); return False

def halaman(url):
    if not boleh_akses(url):
        print('SKIP (robots / domain tidak diizinkan):',url); return None
    try:
        time.sleep(TUNDA_DETIK)
        r = SESSION.get(url, timeout=30)
        if r.status_code in (403, 429, 503) or any(x in r.text[:5000].lower() for x in ('captcha','cloudflare','verify you are human')):
            print('Dibatasi oleh server, berhenti untuk halaman:',r.status_code,url); return None
        r.raise_for_status()
        return BeautifulSoup(r.text, 'html.parser')
    except requests.RequestException as exc:
        print('Gagal akses:',str(exc)[:180]); return None

def link_internal(soup, sumber):
    hasil=[]
    for a in soup.select('a[href]'):
        href=urljoin(sumber,a.get('href','')).split('#')[0]
        p=urlparse(href)
        if p.scheme=='https' and p.netloc=='putusan3.mahkamahagung.go.id' and href not in hasil:
            hasil.append(href)
    return hasil

def temukan_detail(soup, halaman_indeks):
    seen = OrderedDict()
    for a in soup.select('a[href]'):
        href = urljoin(halaman_indeks, a.get('href','')).split('#')[0]
        if '/direktori/putusan/' in urlparse(href).path and urlparse(href).netloc == 'putusan3.mahkamahagung.go.id':
            seen[href] = 1
    return list(seen)

def detail_sesuai_target(soup):
    teks=' '.join(soup.stripped_strings).lower()
    pn=bool(re.search(r'(pn|pengadilan negeri)\s+tanjung\s+rede[bp]|/pn\s*tnr',teks,re.I))
    kategori=('narkotika' in teks or 'psikotropika' in teks)
    return pn and kategori

def temukan_pdf(soup, url_detail):
    urls=[]
    for a in soup.select('a[href]'):
        href=urljoin(url_detail,a.get('href',''))
        path=urlparse(href).path.lower()
        if ('download_file' in path or path.endswith('.pdf')) and urlparse(href).netloc == 'putusan3.mahkamahagung.go.id':
            if href not in urls: urls.append(href)
    return urls

def unduh_pdf(url, name_hint):
    if not boleh_akses(url): return None
    try:
        time.sleep(TUNDA_DETIK)
        r=SESSION.get(url,timeout=60,stream=True)
        if r.status_code != 200:
            print('PDF tidak bisa diunduh:',r.status_code,url); return None
        chunks=[]; total=0
        for block in r.iter_content(1024*256):
            if not block: continue
            chunks.append(block);total+=len(block)
            if total > 30_000_000:
                print('Berkas terlalu besar; dilewati');return None
        body=b''.join(chunks)
        if body[:5]!=b'%PDF-':
            print('Respons bukan PDF asli; dilewati:',url); return None
        suffix=hashlib.sha256(body).hexdigest()[:10]
        dest=PDF_DIR / ('putusan_' + re.sub(r'[^A-Za-z0-9_-]+','_',name_hint)[:70]+'_'+suffix+'.pdf')
        dest.write_bytes(body)
        return dest
    except requests.RequestException as exc:
        print('Gagal unduh:', str(exc)[:150]);return None

def crawl_daftar(start_url):
    antrian=[start_url, *TARGET_URLS]
    dikunjungi=set(); detail=[]
    while antrian and len(dikunjungi)<BATAS_HALAMAN_NAVIGASI and len(detail)<BATAS_DOKUMEN_CRAWLER:
        url=antrian.pop(0)
        if url in dikunjungi: continue
        dikunjungi.add(url)
        soup=halaman(url)
        if soup is None: continue
        detail.extend(x for x in temukan_detail(soup,url) if x not in detail)
        navigasi=[]
        for href in link_internal(soup,url):
            path=urlparse(href).path.lower()
            if href in dikunjungi or '/direktori/putusan/' in path: continue
            if path=='/direktori.html' or path.startswith('/direktori/index/'):
                skor=sum(k in href.lower() for k in ('tanjung','redep','redeb','narkotika','psikotropika','2023','2024','2025'))
                navigasi.append((skor,href))
        for _,href in sorted(navigasi,key=lambda x:(-x[0],x[1]))[:12]:
            if href not in antrian: antrian.append(href)
        print('Navigasi:',len(dikunjungi),'| detail ditemukan:',len(detail))
    return detail[:BATAS_DOKUMEN_CRAWLER]

log_crawler=[]
if AKTIFKAN_CRAWL:
    print('Mulai crawler dari:',START_URL)
    ditemukan=crawl_daftar(START_URL)
    for i,url in enumerate(ditemukan,1):
        soup=halaman(url)
        sesuai=bool(soup and detail_sesuai_target(soup))
        candidates=temukan_pdf(soup,url) if sesuai else []
        result=next((str(x) for href in candidates if (x:=unduh_pdf(href,f'pn_tnr_{i:03d}')) is not None), '')
        status='TERUNDUH' if result else ('TIDAK SESUAI FILTER' if soup and not sesuai else 'PDF TIDAK TERSEDIA/GAGAL')
        log_crawler.append({'start_url':START_URL,'detail_url':url,'sesuai_target':sesuai,'pdf_url':candidates[0] if candidates else '', 'downloaded_pdf':result,'status':status})
    pd.DataFrame(log_crawler,columns=['start_url','detail_url','sesuai_target','pdf_url','downloaded_pdf','status']).to_csv(OUT_DIR/'log_crawling.csv',index=False,encoding='utf-8-sig')
    print('Detail:',len(ditemukan),'| PDF crawler berhasil:',sum(bool(x['downloaded_pdf']) for x in log_crawler))
else:
    print('Crawler nonaktif. Mode pemrosesan PDF manual siap digunakan.')

Crawler nonaktif. Mode pemrosesan PDF manual siap digunakan.


## 3B. Crawler browser Playwright (utama)
Crawler membuka Chrome terlihat, dimulai dari `https://putusan3.mahkamahagung.go.id/direktori.html`, lalu mengunduh putusan PN Tanjung Redeb kategori narkotika/psikotropika. Jika verifikasi muncul, selesaikan sendiri di browser. Proteksi situs tidak dilewati otomatis.

In [ ]:
JALANKAN_CRAWLER_PLAYWRIGHT = True
if JALANKAN_CRAWLER_PLAYWRIGHT:
    import importlib.util
    script = (PROJECT_ROOT / '03_SUBCPMK3_Crawler_dan_Ekstraksi' / 'crawler_ma_playwright.py') if PROJECT_ROOT else Path('crawler_ma_playwright.py')
    if not script.is_file():
        print('Script crawler tidak ditemukan:', script)
        print('Upload crawler_ma_playwright.py bersama notebook atau nonaktifkan crawler Playwright.')
    else:
        spec = importlib.util.spec_from_file_location('crawler_ma_playwright', script)
        crawler_ma = importlib.util.module_from_spec(spec); spec.loader.exec_module(crawler_ma)
        HASIL_CRAWLER = crawler_ma.run_in_notebook(ROOT, quotas={2025:16, 2024:17, 2023:17}, skip_bht=True)
        print('Baris log crawler:', len(HASIL_CRAWLER))


Titik masuk berhasil dibuka: https://putusan3.mahkamahagung.go.id/direktori.html
Tahun 2025: 8 halaman, target 16 PDF
Verifikasi situs muncul. Selesaikan secara manual di browser; menunggu maksimal 5 menit...
Verifikasi selesai.
2025: 1/16 | Putusan_250_Pid.Sus_2025_PN_Tnr.pdf | BELUM DIVERIFIKASI
2025: 2/16 | Putusan_251_Pid.Sus_2025_PN_Tnr.pdf | BELUM DIVERIFIKASI
Verifikasi situs muncul. Selesaikan secara manual di browser; menunggu maksimal 5 menit...


## 4. Ekstraksi teks lengkap PDF → TXT
Teks penuh disimpan tanpa memotong halaman, supaya SUBCPMK 4 nanti memakai seluruh bagian putusan. Berkas hasil scan tanpa teks diberi tanda untuk pemeriksaan OCR, tidak dibuatkan isi secara fiktif.

In [ ]:
def pdf_to_text(path):
    try:
        with fitz.open(path) as doc:
            return '\n'.join(page.get_text('text',sort=True) for page in doc)
    except Exception as exc:
        print('PDF gagal dibaca:',path.name,str(exc)[:100]); return ''

CRAWLER_PDF_DIR = ROOT / 'pdf_crawler'
ACTIVE_PDF_DIR = CRAWLER_PDF_DIR if any(CRAWLER_PDF_DIR.glob('*.pdf')) else PDF_DIR
ACTIVE_TXT_DIR = (ROOT / 'txt_crawler') if ACTIVE_PDF_DIR == CRAWLER_PDF_DIR else TXT_DIR
ACTIVE_TXT_DIR.mkdir(parents=True, exist_ok=True)
print('Sumber PDF yang diproses:', ACTIVE_PDF_DIR)
audit=[]
for pdf in sorted(ACTIVE_PDF_DIR.glob('*.pdf')):
    if pdf.read_bytes()[:5] != b'%PDF-':
        audit.append({'file':pdf.name,'status':'BUKAN PDF','jumlah_karakter':0,'sha256':''})
        continue
    teks=pdf_to_text(pdf)
    try:
        with fitz.open(pdf) as doc: jumlah_halaman=len(doc)
    except Exception: jumlah_halaman=0
    ringkasan_satu_halaman=(jumlah_halaman==1 and 'Barang Bukti' in teks and 'Amar Putusan' in teks)
    (ACTIVE_TXT_DIR/(pdf.stem+'.txt')).write_text(teks,encoding='utf-8')
    audit.append({'file':pdf.name,'status':'TEKS ADA' if len(teks.strip())>100 else 'PERIKSA SCAN/OCR',
                  'jumlah_karakter':len(teks),'jumlah_halaman':jumlah_halaman,'ringkasan_1_halaman':ringkasan_satu_halaman,'sha256':hashlib.sha256(pdf.read_bytes()).hexdigest()})
pd.DataFrame(audit,columns=['file','status','jumlah_karakter','jumlah_halaman','ringkasan_1_halaman','sha256']).to_csv(OUT_DIR/'audit_pdf.csv',index=False)
print('PDF terbaca:',len(audit),'| TXT:',len(list(ACTIVE_TXT_DIR.glob('*.txt'))))

PDF terbaca: 50 | TXT: 50


## 5. Ekstraksi kolom dan audit kualitas sumber

Jika PDF resmi lengkap memiliki judul `MENGADILI`, gunakan ekstraksi bagian amar; untuk tabel 1 halaman, kolom diambil dari posisi teks PDF sebagai fallback. Sumber ringkasan tetap ditandai perlu verifikasi, bukan dianggap valid secara hukum.


In [ ]:
NOMOR_RE = re.compile(r'(?<!\d)(\d{1,5})\s*/\s*Pid\s*\.\s*Sus(?:[-.]\w+)?\s*/\s*(20\d{2})\s*/\s*PN\s*\.?\s*Tnr', re.I)
AMAR_RE = re.compile(r'(?im)^\s*M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?\s*$|^\s*MENGADILI\s*:?\s*$')
PENUTUP_RE = re.compile(r'(?im)^\s*Demikian(?:lah)?\s+(?:di)?putuskan|^\s*Demikian\s+diputuskan')
BARANG_RE = re.compile(r'(?im)^\s*(?:Menetapkan\s+)?barang\s+bukti\s*(?:berupa)?\s*:?' )

def ambil_amar_full(text):
    matches=list(AMAR_RE.finditer(text))
    if not matches:return ''
    body=text[matches[-1].end():]
    penutup=PENUTUP_RE.search(body)
    if penutup:body=body[:penutup.start()]
    return body.strip()

def ambil_bukti_full(amar, text):
    for raw in (amar,text):
        m=list(BARANG_RE.finditer(raw))
        if not m:continue
        body=raw[m[-1].end():]
        s=re.search(r'(?im)^\s*(?:Membebankan|Memerintahkan|Menetapkan\s+(?:agar|terdakwa|masa)|Menghukum|Menjatuhkan|Demikian\s+diputuskan)\b',body)
        return (body[:s.start()] if s else body[:4500]).strip()
    return ''

def baca_kolom_ringkasan(pdf_path):
    """Fallback KHUSUS PDF 1 halaman bertabel, berdasarkan posisi kolom PDF."""
    try:
        with fitz.open(pdf_path) as pdf:
            if len(pdf)!=1:return '', '', False
            page=pdf[0]
            whole=page.get_text('text')
            if not ('Barang Bukti' in whole and 'Amar Putusan' in whole):
                return '', '', False
            # Tata letak template ZIP terakhir: bukti x=249–395, amar x=399–548;
            # jika sumber PDF lain gunakan ekstraksi full-text atau review manual.
            bukti=page.get_text('text',clip=fitz.Rect(242,230,395,page.rect.height-15),sort=True).strip()
            amar=page.get_text('text',clip=fitz.Rect(395,230,565,page.rect.height-15),sort=True).strip()
            return bukti,amar,True
    except Exception as exc:
        print('Gagal baca kolom:',pdf_path.name,exc)
        return '', '', False

rows=[];review=[]
for i,pdf in enumerate(sorted(ACTIVE_PDF_DIR.glob('*.pdf')),1):
    text=pdf_to_text(pdf)
    no=NOMOR_RE.search(text[:30000])
    nomor=f'{no.group(1)}/Pid.Sus/{no.group(2)}/PN Tnr' if no else ''
    amar=ambil_amar_full(text)
    bukti=ambil_bukti_full(amar,text) if amar else ''
    b_kol,a_kol,is_summary=baca_kolom_ringkasan(pdf)
    if is_summary:
        bukti,amar=b_kol,a_kol
    lembaga='PN Tanjung Redeb' if nomor and re.search(r'pengadilan\s+negeri\s+tanjung\s+rede[bp]',text,re.I) else ''
    rows.append({'No':i,'No Putusan':nomor,'Lembaga Peradilan':lembaga,'Barang Bukti':bukti,'Amar Putusan':amar})
    alasan=[]
    if is_summary:alasan.append('PDF RINGKASAN 1 HALAMAN; BUKAN SALINAN PUTUSAN LENGKAP YANG SUDAH TERVERIFIKASI')
    if not nomor:alasan.append('NOMOR TIDAK TERDETEKSI')
    if not lembaga:alasan.append('LEMBAGA TIDAK TERDETEKSI')
    if not bukti:alasan.append('BARANG BUKTI KOSONG')
    if not amar:alasan.append('AMAR PUTUSAN KOSONG')
    alasan.append('STATUS BHT BELUM DIVERIFIKASI DI DIREKTORI MA')
    review.append({'file':pdf.name,'no_putusan':nomor,'tipe_pdf':'RINGKASAN' if is_summary else 'LAINNYA','alasan':'; '.join(alasan)})

hasil=pd.DataFrame(rows,columns=['No','No Putusan','Lembaga Peradilan','Barang Bukti','Amar Putusan'])
hasil.to_csv(OUT_DIR/'hasil_ekstraksi.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(review).to_csv(OUT_DIR/'perlu_review.csv',index=False,encoding='utf-8-sig')
print('PDF terbaca:',len(rows),'| Barang bukti terisi:',hasil['Barang Bukti'].ne('').sum(),'| Amar terisi:',hasil['Amar Putusan'].ne('').sum())
print('PERINGATAN: keberhasilan ekstraksi bukan verifikasi keaslian ataupun status non-BHT.')
if rows: display(hasil[['No','No Putusan','Lembaga Peradilan']].head())


PDF terbaca: 50 | Barang bukti terisi: 50 | Amar terisi: 50
PERINGATAN: keberhasilan ekstraksi bukan verifikasi keaslian ataupun status non-BHT.


,No,No Putusan,Lembaga Peradilan
0,1,102/Pid.Sus/2025/PN Tnr,PN Tanjung Redeb
1,2,105/Pid.Sus/2023/PN Tnr,PN Tanjung Redeb
2,3,108/Pid.Sus/2024/PN Tnr,PN Tanjung Redeb
3,4,111/Pid.Sus/2025/PN Tnr,PN Tanjung Redeb
4,5,114/Pid.Sus/2023/PN Tnr,PN Tanjung Redeb


## 6. Unduh hasil dan validasi
Unduhan ZIP dibuat hanya dari berkas yang benar-benar ada. `perlu_review.csv` dan `audit_pdf.csv` perlu diperiksa sebelum mengumpulkan tugas. **Status BHT tidak dapat disimpulkan dari nomor perkara atau isi amar saja**, harus dicek pada halaman resmi perkara.

In [ ]:
files_for_zip=[p for folder in (ACTIVE_PDF_DIR,ACTIVE_TXT_DIR,OUT_DIR) for p in folder.iterdir() if p.is_file()]
if not any(ACTIVE_PDF_DIR.glob('*.pdf')):
    print('Belum ada PDF nyata. Tidak dibuat ZIP hasil kosong.')
else:
    zpath=ROOT/'Hasil_SUBCPMK3_PN_Tanjung_Redeb.zip'
    with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
        for f in files_for_zip:z.write(f,f.relative_to(ROOT))
    print('ZIP:',zpath,'| ukuran:',round(zpath.stat().st_size/1024/1024,2),'MB')
    if Path('/content').exists():
        print('Untuk mengunduh lewat Colab:')
        print('from google.colab import files; files.download(str(zpath))')
print('Selesai. Periksa review dan status BHT secara manual sebelum unggah.')

ZIP: C:\Users\marce\KULIAH\Smt7\InformationRetrival\UTS\03_SUBCPMK3_Crawler_dan_Ekstraksi\PN_Tanjung_Redeb\Hasil_SUBCPMK3_PN_Tanjung_Redeb.zip | ukuran: 0.19 MB
Selesai. Periksa review dan status BHT secara manual sebelum unggah.
